# Technologies Used — Meeting-Room Booking Chatbot

This notebook walks through every major technology in the project with concrete code examples taken directly from the implementation.  
Each section shows *what* the technology does and *how* it is applied in this solution.

---

## 1. FastAPI — REST API + SPA serving

FastAPI is the web framework for the Python backend.  
Key features used:
- **Typed route parameters** validated by Pydantic automatically
- **`lifespan` context manager** (modern alternative to deprecated `on_event`)
- **CORS middleware** to allow the Vite dev server on port 5173
- **Static file mounting** so the same server delivers the React build in production

In [ ]:
# backend/app/main.py  (abridged)
from contextlib import asynccontextmanager
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse
from pathlib import Path

# The lifespan replaces the old @app.on_event('startup') pattern.
# Everything before `yield` runs on startup; after `yield` on shutdown.
@asynccontextmanager
async def lifespan(app: FastAPI):
    init_db()   # create SQLite tables if they don't exist
    yield

app = FastAPI(title="Cubo Itau Booking Chatbot", version="0.1.0", lifespan=lifespan)

# Allow the Vite dev server to call the API without browser CORS errors.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173", "http://127.0.0.1:5173"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# In production the React SPA is built into frontend/dist and served here.
_FRONTEND_DIST = Path(__file__).resolve().parent.parent.parent / "frontend" / "dist"
if _FRONTEND_DIST.is_dir():
    app.mount("/assets", StaticFiles(directory=_FRONTEND_DIST / "assets"), name="assets")

    @app.get("/{full_path:path}")
    def spa(full_path: str) -> FileResponse:
        """Return index.html for any non-API path (client-side routing)."""
        candidate = _FRONTEND_DIST / full_path
        if candidate.is_file():
            return FileResponse(candidate)
        return FileResponse(_FRONTEND_DIST / "index.html")

In [ ]:
# backend/app/api/bookings.py  (abridged)
# FastAPI routers group related endpoints; Pydantic schemas validate request/response.
from fastapi import APIRouter, Depends, HTTPException, status
from pydantic import BaseModel
from datetime import datetime

router = APIRouter(prefix="/api")

class BookingIn(BaseModel):
    room: str
    title: str
    attendees: int
    start: datetime
    end: datetime

@router.post("/bookings", status_code=status.HTTP_201_CREATED)
def create_booking(
    body: BookingIn,
    username: str = Depends(get_current_user),          # JWT guard
    service: BookingService = Depends(get_booking_service),
):
    try:
        booking = service.create_booking(
            room=body.room, title=body.title, attendees=body.attendees,
            start=body.start, end=body.end, owner=username,
        )
        return booking
    except BookingError as exc:
        raise HTTPException(status_code=400, detail=str(exc))

---
## 2. LangChain + langchain-openai — LLM Tool-Calling

LangChain provides the abstraction layer between the FastAPI backend and the OpenAI API.  
The core pattern used is **tool-calling**: the LLM receives a list of tool schemas, decides which tool(s) to invoke, and the application executes them and feeds the results back.

In [ ]:
# backend/app/agent/chatbot.py  (abridged)
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage, BaseMessage

MAX_TOOL_ITERATIONS = 6

def run_chat(*, user_message: str, history: list[dict], service, username: str) -> str:
    # Build the tool list, closing over the acting user (security: LLM cannot pick a different owner).
    tools = build_tools(service, username)
    tools_by_name = {t.name: t for t in tools}

    # bind_tools() adds the tool schemas to every request so the model can choose to call them.
    llm = ChatOpenAI(
        model=settings.openai_model,   # gpt-4.1-mini
        api_key=settings.openai_api_key,
        temperature=0,                 # deterministic for booking logic
    ).bind_tools(tools)

    # Reconstruct conversation history as LangChain message objects.
    messages: list[BaseMessage] = [SystemMessage(content=_system_prompt(username))]
    for turn in history:
        if turn["role"] == "user":
            messages.append(HumanMessage(content=turn["content"]))
        else:
            messages.append(AIMessage(content=turn["content"]))
    messages.append(HumanMessage(content=user_message))

    # Agentic loop: keep calling the LLM until it produces a final text answer.
    for _ in range(MAX_TOOL_ITERATIONS):
        ai_msg: AIMessage = llm.invoke(messages)
        messages.append(ai_msg)

        if not ai_msg.tool_calls:          # no tools needed → we have the answer
            return ai_msg.content

        # Execute each requested tool and feed results back as ToolMessages.
        for call in ai_msg.tool_calls:
            tool = tools_by_name.get(call["name"])
            try:
                result = tool.invoke(call["args"]) if tool else f"Unknown tool '{call['name']}'"
            except BookingError as exc:
                result = f"Booking error: {exc}"
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

    return "I wasn't able to complete that request after several steps. Could you rephrase it?"

In [ ]:
# backend/app/agent/tools.py  (abridged)
# StructuredTool wraps a plain Python function and derives the JSON schema
# from a Pydantic args model — that schema is what the LLM sees.
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

class CreateBookingArgs(BaseModel):
    room: str       = Field(description="Room letter: A, B, C, D or E.")
    title: str      = Field(description="Title of the meeting.")
    attendees: int  = Field(description="Number of attendees; must not exceed room capacity.")
    start: str      = Field(description="Start datetime, ISO-8601, e.g. 2026-08-20T10:00.")
    end: str        = Field(description="End datetime, ISO-8601. Max 3h after start, 30-min aligned.")

def build_tools(service, owner: str) -> list[StructuredTool]:
    """Factory: each returned tool closes over `owner` so the LLM cannot act as another user."""

    def create_booking(room: str, title: str, attendees: int, start: str, end: str) -> str:
        booking = service.create_booking(
            room=room, title=title, attendees=attendees,
            start=_parse(start), end=_parse(end),
            owner=owner,     # <-- injected from JWT, never from LLM output
        )
        return f"Booked room {booking.room} '{booking.title}' ... (id {booking.id})."

    return [
        StructuredTool.from_function(
            create_booking,
            name="create_booking",
            description="Create a meeting-room booking for the current user.",
            args_schema=CreateBookingArgs,
        ),
        # ... four more tools omitted for brevity
    ]

---
## 3. SQLModel — ORM + Pydantic schema in one class

SQLModel combines SQLAlchemy (database ORM) with Pydantic (data validation).  
A single `class Booking(SQLModel, table=True)` is both the database table definition and the Pydantic model used in API responses — no duplication.

In [ ]:
# backend/app/domain/models.py
from datetime import datetime
from sqlmodel import SQLModel, Field

class Booking(SQLModel, table=True):
    """Database table AND Pydantic validation schema — defined once."""
    id:         int | None  = Field(default=None, primary_key=True)
    room:       str
    title:      str
    attendees:  int
    start:      datetime
    end:        datetime
    owner:      str
    created_at: datetime    = Field(default_factory=datetime.utcnow)

In [ ]:
# backend/app/repositories/database.py
from sqlmodel import SQLModel, create_engine, Session
from app.core.config import settings

# Engine is created once; SQLite by default, swappable to Postgres via DATABASE_URL.
_engine = create_engine(settings.database_url, connect_args={"check_same_thread": False})

def init_db() -> None:
    SQLModel.metadata.create_all(_engine)   # creates tables if they don't exist

def get_session():
    with Session(_engine) as session:
        yield session

In [ ]:
# backend/app/repositories/booking_repo.py  (abridged)
from sqlmodel import Session, select
from app.domain.models import Booking

class BookingRepository:
    def __init__(self, session: Session):
        self.session = session

    def add(self, booking: Booking) -> Booking:
        self.session.add(booking)
        self.session.commit()
        self.session.refresh(booking)
        return booking

    def for_room_in_range(self, room: str, start, end) -> list[Booking]:
        # Half-open interval overlap query: [start, end) overlaps [b.start, b.end)
        return self.session.exec(
            select(Booking).where(
                Booking.room == room,
                Booking.start < end,
                Booking.end > start,
            )
        ).all()

---
## 4. pydantic-settings — Typed configuration from `.env`

`pydantic-settings` reads environment variables (and `.env` files) into a typed Python object.  
This means all configuration is validated at startup, not at the moment a key is first used.

In [ ]:
# backend/app/core/config.py
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    # Reads from .env for local dev, or from real environment variables in production.
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore",   # silently ignore unknown env vars (e.g. Railway's injected vars)
    )

    openai_api_key:               str = ""
    openai_model:                 str = "gpt-4.1-mini"
    jwt_secret:                   str = "change-me-in-production"
    jwt_algorithm:                str = "HS256"
    access_token_expire_minutes:  int = 480   # 8-hour working day
    app_password:                 str = "TechnicalChallengePromtior"
    database_url:                 str = "sqlite:///./booking.db"

settings = Settings()   # singleton — imported everywhere as `from app.core.config import settings`

# Usage example:
print(settings.openai_model)     # "gpt-4.1-mini"
print(settings.database_url)     # "sqlite:///./booking.db"

In [ ]:
# backend/.env.example  — copy to .env and fill in real values
env_example = """
OPENAI_API_KEY=sk-your-key-here
OPENAI_MODEL=gpt-4.1-mini
JWT_SECRET=change-me-in-production
APP_PASSWORD=TechnicalChallengePromtior
DATABASE_URL=sqlite:///./booking.db
"""
print(env_example)

---
## 5. JWT Authentication — python-jose

Authentication is implemented with JSON Web Tokens (JWT).  
The login endpoint returns a signed token; every subsequent request sends it in the `Authorization: Bearer` header.

In [ ]:
# backend/app/core/security.py
from datetime import datetime, timedelta
from jose import jwt, JWTError
from app.core.config import settings

USERS = {"User1", "User2"}

def authenticate(username: str, password: str) -> bool:
    """Returns True only if username is known AND password matches the shared secret."""
    return username in USERS and password == settings.app_password

def create_access_token(username: str) -> str:
    payload = {
        "sub": username,
        "exp": datetime.utcnow() + timedelta(minutes=settings.access_token_expire_minutes),
    }
    return jwt.encode(payload, settings.jwt_secret, algorithm=settings.jwt_algorithm)

def decode_token(token: str) -> str | None:
    """Returns the username if the token is valid, otherwise None."""
    try:
        payload = jwt.decode(token, settings.jwt_secret, algorithms=[settings.jwt_algorithm])
        return payload.get("sub")
    except JWTError:
        return None

In [ ]:
# backend/app/api/deps.py  — FastAPI dependency that guards every protected endpoint
from fastapi import Depends, HTTPException, status
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from app.core.security import decode_token

_bearer = HTTPBearer()

def get_current_user(
    credentials: HTTPAuthorizationCredentials = Depends(_bearer),
) -> str:
    username = decode_token(credentials.credentials)
    if username is None:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid or expired token",
        )
    return username   # injected as a parameter into every route that declares Depends(get_current_user)

---
## 6. Pure Domain Rules — testable business logic

The booking validation is entirely pure Python (no database, no LLM).  
This makes it trivial to unit-test every edge case in isolation.

In [ ]:
# backend/app/domain/booking_rules.py  (abridged)
from datetime import datetime, timedelta

SLOT_MINUTES = 30
MAX_DURATION = timedelta(hours=3)

class BookingError(ValueError):
    """Raised when a booking violates a business rule; surfaced as a user-facing message."""

def overlaps(start_a: datetime, end_a: datetime, start_b: datetime, end_b: datetime) -> bool:
    """Half-open interval overlap: [start, end). Back-to-back meetings do NOT overlap."""
    return start_a < end_b and start_b < end_a

# Quick demonstration
t = datetime(2026, 8, 20, 10, 0)
print(overlaps(t, t + timedelta(hours=1.5),
               t + timedelta(hours=1.5), t + timedelta(hours=2)))  # False — back-to-back
print(overlaps(t, t + timedelta(hours=1.5),
               t + timedelta(hours=1), t + timedelta(hours=2)))    # True — genuine conflict

In [ ]:
# The 17 unit tests all run against booking_rules.py with no database needed.
# Example from backend/tests/test_booking_rules.py:

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname('__file__'), '..', 'backend'))

from datetime import datetime, timedelta

def overlaps(start_a, end_a, start_b, end_b):
    return start_a < end_b and start_b < end_a

base = datetime(2026, 8, 20, 9, 0)

# Test: back-to-back meetings do not conflict
assert not overlaps(
    base, base + timedelta(hours=1),
    base + timedelta(hours=1), base + timedelta(hours=2)
), "Back-to-back should NOT overlap"

# Test: genuine overlap is detected
assert overlaps(
    base, base + timedelta(minutes=90),
    base + timedelta(minutes=60), base + timedelta(minutes=120)
), "Overlapping intervals should be detected"

print("All overlap assertions passed.")

---
## 7. React + Vite — Frontend SPA

The frontend is a single-page application built with React 18 and bundled by Vite.  
Vite's dev server proxies `/api` calls to the FastAPI backend, so no CORS issues during development.

### Component tree
```
App
 ├── Login       (username dropdown + password field → POST /api/login → JWT stored in localStorage)
 ├── Chat        (message history, typing indicator → POST /api/chat)
 └── ScheduleGrid (room/date picker → GET /api/rooms/{room}/schedule, green=free, red=occupied)
```

In [ ]:
# frontend/vite.config.js — the proxy is the key dev-time feature
vite_config = """
import { defineConfig } from 'vite'
import react from '@vitejs/plugin-react'

export default defineConfig({
  plugins: [react()],
  server: {
    proxy: {
      // All fetch('/api/...') calls from the browser go to the FastAPI server.
      '/api': 'http://localhost:8000',
    },
  },
})
"""
print(vite_config)

In [ ]:
# frontend/src/api.js — thin fetch wrapper that attaches the stored JWT
api_js = """
const BASE = ''   // requests go through Vite proxy in dev, same origin in prod

function authHeaders() {
  const token = localStorage.getItem('token')
  return token ? { Authorization: `Bearer ${token}` } : {}
}

export async function login(username, password) {
  const res = await fetch(`${BASE}/api/login`, {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify({ username, password }),
  })
  if (!res.ok) throw new Error('Login failed')
  const { access_token } = await res.json()
  localStorage.setItem('token', access_token)
  localStorage.setItem('username', username)
}

export async function sendMessage(message, history) {
  const res = await fetch(`${BASE}/api/chat`, {
    method: 'POST',
    headers: { 'Content-Type': 'application/json', ...authHeaders() },
    body: JSON.stringify({ message, history }),
  })
  if (!res.ok) throw new Error('Chat request failed')
  return res.json()   // { reply: "..." }
}
"""
print(api_js)

---
## 8. Docker — Multi-stage build for single-image deployment

A multi-stage Dockerfile compiles the React app in a Node container, then copies the result into the Python container.  
The final image contains only the Python runtime — no Node.js — and serves both the API and the SPA.

In [ ]:
dockerfile_content = """
# --- Stage 1: build the React SPA -------------------------------------------
FROM node:22-alpine AS frontend
WORKDIR /frontend
COPY frontend/package*.json ./
RUN npm ci                     # clean install — reproducible
COPY frontend/ ./
RUN npm run build              # -> /frontend/dist

# --- Stage 2: Python backend serving API + the built SPA --------------------
FROM python:3.12-slim
ENV PYTHONUNBUFFERED=1 PYTHONDONTWRITEBYTECODE=1 PIP_NO_CACHE_DIR=1
WORKDIR /app

# Install backend deps (pyproject.toml declares all dependencies).
COPY backend/pyproject.toml /app/backend/pyproject.toml
COPY backend/app            /app/backend/app
RUN pip install /app/backend

# Copy the compiled React app where main.py expects it.
COPY --from=frontend /frontend/dist /app/frontend/dist

WORKDIR /app/backend
EXPOSE 8000
CMD uvicorn app.main:app --host 0.0.0.0 --port ${PORT:-8000}
"""
print(dockerfile_content)

In [ ]:
# railway.json — tells Railway to use the Dockerfile and where the health check is
import json

railway_config = {
    "$schema": "https://railway.app/railway.schema.json",
    "build": {
        "builder": "DOCKERFILE",
        "dockerfilePath": "Dockerfile"
    },
    "deploy": {
        "startCommand": "uvicorn app.main:app --host 0.0.0.0 --port $PORT",
        "healthcheckPath": "/api/health",
        "healthcheckTimeout": 100,
        "restartPolicyType": "ON_FAILURE",
        "restartPolicyMaxRetries": 3
    }
}
print(json.dumps(railway_config, indent=2))

---
## 9. Architecture patterns summary

| Pattern | Where | Why |
|---|---|---|
| **Layered architecture** | domain → repo → service → api/agent | Each layer has a single responsibility and one direction of dependency |
| **Tool factory / closure** | `build_tools(service, owner)` | LLM cannot choose whose data it acts on — ownership comes from the JWT |
| **Half-open intervals** | `overlaps()` in `booking_rules.py` | Back-to-back meetings share a boundary without conflicting |
| **Pure domain rules** | `booking_rules.py` | No DB or LLM needed → 17 unit tests, instant feedback |
| **Single entry point for business logic** | `BookingService` | REST API and LLM tools both call the same service → identical behaviour |
| **pydantic-settings singleton** | `settings = Settings()` | Validated config at import time, available everywhere via one import |
| **FastAPI lifespan** | `@asynccontextmanager async def lifespan` | Modern startup/shutdown pattern (replaces deprecated `on_event`) |
| **Multi-stage Docker** | `Dockerfile` | Tiny final image (no Node); reproducible build; single service to deploy |